# Assignment: Slippage & Costs — Transaction Cost Models & Liquidity Premium
**Navigating the Hidden Friction Between Research and Production**

| Part | Focus | Methods |
|------|-------|---------|
| 1 | Cost Taxonomy & Decomposition | Explicit vs implicit, spread, slippage, market impact |
| 2 | The Modeling Spectrum (Levels 1–4) | Flat fee, % model, bid-ask spread, square-root |
| 3 | Almgren-Chriss Optimal Execution | Temporary vs permanent impact, execution frontier |
| 4 | Liquidity Premium | Spread-sorted portfolios, illiquidity discount |
| 5 | Optimal Urgency & Capacity | POV sweep, alpha decay vs market impact trade-off |

> *All data is fully simulated; no external feeds required.*

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
import plotly.graph_objects as go
import plotly.subplots as ps
import warnings
warnings.filterwarnings('ignore')

SEED   = 42
rng    = np.random.default_rng(SEED)
ANNUAL = 252

# ── Simulate a universe of 4 stocks with varying liquidity ────────────────────
T     = 504   # ~2 years
dates = pd.date_range('2023-01-02', periods=T, freq='B')

# Large-cap liquid | Large-cap illiquid | Small-cap liquid | Small-cap illiquid
UNIVERSE = {
    'LC_Liquid':   dict(mu=0.0002, vol=0.010, adv=2_000_000, spread_bps=2,  price=150.0),
    'LC_Illiquid': dict(mu=0.0003, vol=0.012, adv=  300_000, spread_bps=8,  price=120.0),
    'SC_Liquid':   dict(mu=0.0004, vol=0.018, adv=  150_000, spread_bps=15, price= 45.0),
    'SC_Illiquid': dict(mu=0.0006, vol=0.022, adv=   40_000, spread_bps=35, price= 18.0),
}

prices, rets, volumes, spreads_bps = {}, {}, {}, {}
for name, p in UNIVERSE.items():
    r = p['mu'] + p['vol'] * rng.standard_normal(T)
    rets[name]       = r
    prices[name]     = p['price'] * np.cumprod(1 + r)
    spreads_bps[name]= p['spread_bps'] * (1 + 0.5 * np.abs(r) / p['vol'])  # spread widens with vol
    volumes[name]    = rng.lognormal(np.log(p['adv']), 0.4, T)

price_df  = pd.DataFrame(prices,  index=dates)
ret_df    = pd.DataFrame(rets,    index=dates)
vol_df    = pd.DataFrame(volumes, index=dates)
sprd_df   = pd.DataFrame(spreads_bps, index=dates)

print('Universe summary:')
for name, p in UNIVERSE.items():
    ann_ret = p['mu'] * ANNUAL
    ann_vol = p['vol'] * ANNUAL**0.5
    print(f'  {name:<14}  mu={ann_ret:.1%}/yr  vol={ann_vol:.1%}/yr  '
          f'ADV={p["adv"]:>9,}  spread={p["spread_bps"]}bps  price=${p["price"]}')

Universe summary:
  LC_Liquid       mu=5.0%/yr  vol=15.9%/yr  ADV=2,000,000  spread=2bps  price=$150.0
  LC_Illiquid     mu=7.6%/yr  vol=19.0%/yr  ADV=  300,000  spread=8bps  price=$120.0
  SC_Liquid       mu=10.1%/yr  vol=28.6%/yr  ADV=  150,000  spread=15bps  price=$45.0
  SC_Illiquid     mu=15.1%/yr  vol=34.9%/yr  ADV=   40,000  spread=35bps  price=$18.0


---
## Part 1 | Cost Taxonomy & Decomposition

Total trading friction = **Explicit** (deterministic) + **Implicit** (stochastic):

$$\text{Total Cost} = \underbrace{C_{\text{commission}} + C_{\text{fees}}}_{\text{Explicit}} + \underbrace{\frac{1}{2}\text{Spread} + \Delta P_{\text{impact}} + C_{\text{opportunity}}}_{\text{Implicit}}$$

We simulate 1,000 trades on each stock and decompose the friction into its four components.

In [2]:
# ── Cost decomposition per stock ──────────────────────────────────────────────
COMMISSION_PER_SHARE = 0.005     # $0.005 / share (explicit)
EXCHANGE_FEE_BPS     = 0.30      # 0.30 bps (explicit)
SQRT_ETA             = 0.10      # square-root impact coefficient
ORDER_NOTIONAL       = 50_000    # $50k per order

rows = []
for name, p in UNIVERSE.items():
    px      = p['price']
    shares  = ORDER_NOTIONAL / px
    adv_val = p['adv'] * px          # ADV in $
    participation = shares / p['adv']

    # Explicit
    c_commission = shares * COMMISSION_PER_SHARE
    c_exchange   = ORDER_NOTIONAL * EXCHANGE_FEE_BPS / 10_000

    # Implicit: half-spread
    c_spread = ORDER_NOTIONAL * (p['spread_bps'] / 2) / 10_000

    # Implicit: square-root market impact
    c_impact = SQRT_ETA * p['vol'] * np.sqrt(participation) * ORDER_NOTIONAL

    total    = c_commission + c_exchange + c_spread + c_impact
    total_bps = total / ORDER_NOTIONAL * 10_000

    rows.append(dict(
        stock=name,
        commission_bps = c_commission / ORDER_NOTIONAL * 10_000,
        exchange_bps   = EXCHANGE_FEE_BPS,
        spread_bps     = p['spread_bps'] / 2,
        impact_bps     = c_impact / ORDER_NOTIONAL * 10_000,
        total_bps      = total_bps,
        participation  = participation * 100
    ))

cost_df = pd.DataFrame(rows).set_index('stock')
print(f'Order notional: ${ORDER_NOTIONAL:,}  |  per-stock decomposition (bps):\n')
print(cost_df[['commission_bps','exchange_bps','spread_bps','impact_bps','total_bps','participation']]
      .rename(columns={'commission_bps':'Comm','exchange_bps':'Exch',
                        'spread_bps':'½Spread','impact_bps':'Impact',
                        'total_bps':'Total','participation':'POV%'})
      .to_string(float_format='{:.2f}'.format))

Order notional: $50,000  |  per-stock decomposition (bps):

             Comm  Exch  ½Spread  Impact  Total  POV%
stock                                                
LC_Liquid    0.33  0.30     1.00    0.13   1.76  0.02
LC_Illiquid  0.42  0.30     4.00    0.45   5.16  0.14
SC_Liquid    1.11  0.30     7.50    1.55  10.46  0.74
SC_Illiquid  2.78  0.30    17.50    5.80  26.38  6.94


In [3]:
# ── Stacked bar: cost decomposition ───────────────────────────────────────────
components = ['commission_bps', 'exchange_bps', 'spread_bps', 'impact_bps']
labels     = ['Commission', 'Exchange Fee', '½ Bid-Ask Spread', 'Market Impact']
colors     = ['#4a6fa5', '#00d4aa', '#f0c040', '#ff6b6b']

fig = go.Figure()
for col, label, color in zip(components, labels, colors):
    fig.add_trace(go.Bar(
        x=cost_df.index, y=cost_df[col],
        name=label, marker_color=color
    ))

fig.update_layout(
    template='plotly_dark', height=430, barmode='stack',
    title=f'Transaction Cost Decomposition — ${ORDER_NOTIONAL:,} Order per Stock',
    yaxis_title='Cost (basis points)',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

print('Key insight: for the illiquid small-cap, market impact alone exceeds the spread.')
print(f'SC_Illiquid total cost: {cost_df.loc["SC_Illiquid","total_bps"]:.1f} bps '
      f'vs LC_Liquid: {cost_df.loc["LC_Liquid","total_bps"]:.1f} bps')

Key insight: for the illiquid small-cap, market impact alone exceeds the spread.
SC_Illiquid total cost: 26.4 bps vs LC_Liquid: 1.8 bps


**Key observation:** For large-cap liquid stocks, commissions dominate. For small-cap illiquid stocks, market impact and spread dwarf fixed fees — a naive flat-fee model catastrophically underestimates costs.

---
## Part 2 | The Modeling Spectrum (Levels 1–4)

We run the same MA-crossover strategy through four progressively realistic cost models:

| Level | Model | Formula |
|-------|-------|---------|
| 1 | Flat fee | $\$C$ per trade |
| 2 | Percentage | $\alpha\%$ of notional |
| 3 | Bid-ask spread | $\frac{1}{2}(\text{Ask} - \text{Bid})$ per share |
| 4 | Square-root impact | $\eta \cdot \sigma \cdot \sqrt{Q / V_{\text{ADV}}}$ |

Each model is applied on top of the previous, so we can see the **incremental drag** each layer adds.

In [4]:
# ── MA crossover strategy on LC_Illiquid (interesting regime) ─────────────────
FAST, SLOW  = 8, 30
STOCK       = 'LC_Illiquid'
ORDER_QTY   = 500            # shares per trade
p_param     = UNIVERSE[STOCK]

px_series   = price_df[STOCK]
ret_series  = ret_df[STOCK]
vol_series  = vol_df[STOCK]
sprd_series = sprd_df[STOCK]
adv_mean    = p_param['adv']

# Signal (causal, 1-bar lag)
ma_f   = px_series.rolling(FAST).mean()
ma_s   = px_series.rolling(SLOW).mean()
signal = np.sign(ma_f - ma_s).shift(1).fillna(0)
trades = signal.diff().abs().fillna(0) > 0   # True on every direction change

# Raw PnL (no costs)
raw_pnl = signal.shift(1) * ret_series

# ── Level 1: Flat fee ─────────────────────────────────────────────────────────
FLAT_FEE_PER_TRADE = 10.0
notional     = ORDER_QTY * px_series
flat_cost_rt = trades * FLAT_FEE_PER_TRADE / notional
pnl_l1       = raw_pnl - flat_cost_rt

# ── Level 2: % of notional (5 bps) ───────────────────────────────────────────
PCT_BPS      = 5.0
pct_cost_rt  = trades * PCT_BPS / 10_000
pnl_l2       = raw_pnl - flat_cost_rt - pct_cost_rt

# ── Level 3: + bid-ask spread (dynamic, widens with vol) ─────────────────────
spread_cost_rt = trades * (sprd_series / 2) / 10_000
pnl_l3         = pnl_l2 - spread_cost_rt

# ── Level 4: + square-root market impact ─────────────────────────────────────
participation    = ORDER_QTY / vol_series.clip(lower=1)
impact_bps_daily = SQRT_ETA * p_param['vol'] * np.sqrt(participation) * 10_000
impact_cost_rt   = trades * impact_bps_daily / 10_000
pnl_l4           = pnl_l3 - impact_cost_rt

def sharpe(pnl):
    p = pnl.dropna()
    return p.mean() / p.std() * ANNUAL**0.5 if p.std() > 0 else 0

def equity(pnl, base=1.0):
    return base * (1 + pnl.fillna(0)).cumprod()

sr0, sr1, sr2, sr3, sr4 = (sharpe(p) for p in
    [raw_pnl, pnl_l1, pnl_l2, pnl_l3, pnl_l4])

print(f'Strategy: {STOCK}  |  {ORDER_QTY} shares/trade  |  {trades.sum():.0f} trades over {T} bars')
print(f'\n  Raw (no cost):       SR = {sr0:.3f}')
print(f'  + Flat fee:          SR = {sr1:.3f}  (drag={sr0-sr1:.3f})')
print(f'  + % notional:        SR = {sr2:.3f}  (drag={sr1-sr2:.3f})')
print(f'  + Bid-ask spread:    SR = {sr3:.3f}  (drag={sr2-sr3:.3f})')
print(f'  + Market impact:     SR = {sr4:.3f}  (drag={sr3-sr4:.3f})')
print(f'\n  Total SR drag:       {sr0-sr4:.3f} ({(sr0-sr4)/abs(sr0)*100:.0f}% of raw SR)')

Strategy: LC_Illiquid  |  500 shares/trade  |  23 trades over 504 bars

  Raw (no cost):       SR = -0.820
  + Flat fee:          SR = -0.830  (drag=0.010)
  + % notional:        SR = -0.860  (drag=0.030)
  + Bid-ask spread:    SR = -0.895  (drag=0.035)
  + Market impact:     SR = -0.898  (drag=0.003)

  Total SR drag:       0.077 (9% of raw SR)


In [5]:
# ── Equity curves: cost model layers ──────────────────────────────────────────
fig = ps.make_subplots(rows=2, cols=1, vertical_spacing=0.10,
    subplot_titles=['Equity Curves: Progressive Cost Model Layers',
                    'Cumulative Cost Drag (vs Raw)'])

CMAP = ['white', '#4a6fa5', '#00d4aa', '#f0c040', '#ff6b6b']
LABS = [f'Raw (SR={sr0:.2f})', f'L1 Flat fee (SR={sr1:.2f})',
        f'L2 +% notional (SR={sr2:.2f})', f'L3 +Spread (SR={sr3:.2f})',
        f'L4 +Impact (SR={sr4:.2f})']

for pnl, color, lab in zip([raw_pnl, pnl_l1, pnl_l2, pnl_l3, pnl_l4], CMAP, LABS):
    fig.add_trace(go.Scatter(x=dates, y=equity(pnl), mode='lines',
        name=lab, line=dict(color=color, width=2 if color == 'white' else 1.5)),
        row=1, col=1)

fig.add_hline(y=1, line_color='grey', line_dash='dot', row=1, col=1)

# Cumulative drag
for pnl, color, lab in zip([pnl_l1, pnl_l2, pnl_l3, pnl_l4], CMAP[1:], LABS[1:]):
    drag = equity(raw_pnl) - equity(pnl)
    fig.add_trace(go.Scatter(x=dates, y=drag, mode='lines',
        name=lab, line=dict(color=color, width=1.5), showlegend=False),
        row=2, col=1)

fig.update_layout(template='plotly_dark', height=620,
    title=f'The Modeling Spectrum — {STOCK}')
fig.update_yaxes(title_text='Portfolio Value', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Drag ($)', row=2, col=1)
fig.show()

---
## Part 3 | Almgren-Chriss Optimal Execution

The Almgren-Chriss framework decomposes market impact into:

- **Temporary impact** $h(v) = \eta \cdot v^k$ — instantaneous cost from poking the book; mean-reverts
- **Permanent impact** $g(v) = \gamma \cdot v$ — information signal; does not mean-revert

The **optimal execution** problem: liquidate $X$ shares in time $T$ to minimise expected cost + variance:

$$\min_{\{x_k\}} \; \underbrace{\sum_{k} g(v_k) \cdot x_k}_{\text{permanent}} + \underbrace{\sum_{k} h(v_k)}_{\text{temporary}} + \underbrace{\lambda \sigma^2 \sum_k x_k^2}_{\text{risk penalty}}$$

We solve this for a range of risk-aversion parameters $\lambda$ to trace the **execution frontier**.

In [6]:
# ── Almgren-Chriss discretised optimal liquidation ────────────────────────────
X0    = 100_000   # shares to liquidate
N     = 20        # number of time slices (trading periods)
SIGMA = p_param['vol']   # per-period vol
ETA   = 2.5e-7    # temporary impact coefficient ($/share per share/period)
GAMMA = 2.5e-8    # permanent impact coefficient  ($/share per share/period)
K_EXP = 0.6       # exponent for temporary impact (< 1 → concave)

def ac_cost(traj, sigma, eta, gamma, k, lam):
    """
    Compute Almgren-Chriss objective for a given liquidation trajectory.
    traj: array of shares remaining at each period (length N+1, traj[0]=X0, traj[-1]=0)
    Returns (expected_cost, variance, objective)
    """
    trades_k = -np.diff(traj)           # shares sold each period (positive)
    v        = trades_k                 # trading rate = shares per period

    perm_impact = gamma * np.cumsum(v)  # permanent price depression
    temp_impact = eta * v**k            # instantaneous temporary cost

    exp_cost = np.sum(temp_impact * v) + np.sum(perm_impact * v)
    variance = sigma**2 * np.sum(traj[1:]**2)
    objective = exp_cost + lam * variance
    return exp_cost, variance, objective

# Sweep risk-aversion λ → traces the execution frontier
lambdas   = np.logspace(-12, -8, 40)
costs_out, risks_out, trajs_out = [], [], []

for lam in lambdas:
    def objective(x_flat):
        # x_flat: interior inventory levels (length N-1)
        traj = np.concatenate([[X0], x_flat, [0]])
        traj = np.clip(traj, 0, X0)   # no negative or oversize inventory
        _, _, obj = ac_cost(traj, SIGMA, ETA, GAMMA, K_EXP, lam)
        return obj

    # Initialise with TWAP (uniform liquidation)
    x0_vec = np.linspace(X0, 0, N + 1)[1:-1]
    bounds  = [(0, X0)] * (N - 1)
    from scipy.optimize import minimize
    res = minimize(objective, x0_vec, method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 500, 'ftol': 1e-12})

    traj = np.concatenate([[X0], np.clip(res.x, 0, X0), [0]])
    exp_cost, variance, _ = ac_cost(traj, SIGMA, ETA, GAMMA, K_EXP, lam)
    costs_out.append(exp_cost)
    risks_out.append(np.sqrt(variance))   # risk in shares²→ shares unit
    trajs_out.append(traj)

costs_arr = np.array(costs_out)
risks_arr = np.array(risks_out)

# Also compute TWAP and VWAP (aggressive) benchmarks
twap_traj = np.linspace(X0, 0, N + 1)
vwap_traj = X0 * (1 - (np.arange(N + 1) / N)**0.5)  # front-loaded

twap_cost, twap_risk, _ = ac_cost(twap_traj, SIGMA, ETA, GAMMA, K_EXP, 0)
vwap_cost, vwap_risk, _ = ac_cost(vwap_traj, SIGMA, ETA, GAMMA, K_EXP, 0)

print(f'Liquidation: {X0:,} shares over {N} periods')
print(f'TWAP cost: ${twap_cost:,.0f}  |  TWAP timing risk (σ): {twap_risk:,.0f}')
print(f'Aggressive (VWAP) cost: ${vwap_cost:,.0f}  |  risk: {vwap_risk:,.0f}')
print(f'Frontier range: cost ${costs_arr.min():,.0f} – ${costs_arr.max():,.0f}')

Liquidation: 100,000 shares over 20 periods
TWAP cost: $135  |  TWAP timing risk (σ): 8,892,000
Aggressive (VWAP) cost: $141  |  risk: 4,207,876
Frontier range: cost $135 – $135


In [7]:
# ── Execution frontier + sample trajectories ──────────────────────────────────
fig = ps.make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=['Execution Frontier (Cost vs Timing Risk)',
                    'Sample Liquidation Trajectories'])

# Frontier
fig.add_trace(go.Scatter(
    x=risks_arr / X0 * 100, y=costs_arr / X0 * 10_000,   # bps units
    mode='lines+markers', name='AC Frontier',
    marker=dict(color='#00d4aa', size=4),
    line=dict(color='#00d4aa', width=2)), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[twap_risk / X0 * 100], y=[twap_cost / X0 * 10_000],
    mode='markers', name='TWAP',
    marker=dict(color='#f0c040', size=12, symbol='star')), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[vwap_risk / X0 * 100], y=[vwap_cost / X0 * 10_000],
    mode='markers', name='Aggressive (VWAP)',
    marker=dict(color='#ff6b6b', size=12, symbol='diamond')), row=1, col=1)

# Trajectories at low, mid, high risk-aversion
traj_indices = [0, len(lambdas) // 2, -1]
traj_colors  = ['#ff6b6b', '#00d4aa', 'cornflowerblue']
traj_names   = ['Low λ (aggressive)', 'Mid λ', 'High λ (patient)']
periods      = np.arange(N + 1)

for idx, color, name in zip(traj_indices, traj_colors, traj_names):
    fig.add_trace(go.Scatter(
        x=periods, y=trajs_out[idx] / X0 * 100,
        mode='lines+markers', name=name,
        marker=dict(size=4), line=dict(color=color, width=2)), row=1, col=2)

fig.add_trace(go.Scatter(
    x=periods, y=twap_traj / X0 * 100,
    mode='lines', name='TWAP',
    line=dict(color='#f0c040', dash='dot', width=1.5)), row=1, col=2)

fig.update_layout(template='plotly_dark', height=440,
    title='Almgren-Chriss Optimal Execution')
fig.update_xaxes(title_text='Timing Risk (% of inventory)', row=1, col=1)
fig.update_xaxes(title_text='Trading Period', row=1, col=2)
fig.update_yaxes(title_text='Expected Cost (bps)', row=1, col=1)
fig.update_yaxes(title_text='Remaining Inventory (%)', row=1, col=2)
fig.show()

print('Low λ (risk-neutral): front-loads trading → high impact, low timing risk')
print('High λ (risk-averse): spreads trading evenly → low impact, high timing risk')

Low λ (risk-neutral): front-loads trading → high impact, low timing risk
High λ (risk-averse): spreads trading evenly → low impact, high timing risk


In [8]:
# ── Temporary vs permanent impact decomposition ────────────────────────────────
# Visualise price path under temporary+permanent impact for TWAP vs aggressive
N_PATHS = 200

def simulate_impact_path(traj, sigma, eta, gamma, k, n_paths, seed=None):
    rng_loc = np.random.default_rng(seed)
    n       = len(traj) - 1
    v       = -np.diff(traj)                 # trades per period
    perm    = np.concatenate([[0], gamma * np.cumsum(v)])  # permanent shift
    temp    = np.concatenate([[0], eta * v**k])             # temporary shift (instantaneous)

    paths = np.zeros((n_paths, n + 1))
    for i in range(n_paths):
        noise = sigma * rng_loc.standard_normal(n)
        p     = np.zeros(n + 1)
        for t in range(1, n + 1):
            p[t] = p[t-1] + noise[t-1] - perm[t] - temp[t]
        paths[i] = p
    return paths, perm, temp

twap_paths, twap_perm, twap_temp = simulate_impact_path(
    twap_traj, SIGMA, ETA, GAMMA, K_EXP, N_PATHS, seed=1)
aggr_paths, aggr_perm, aggr_temp = simulate_impact_path(
    vwap_traj, SIGMA, ETA, GAMMA, K_EXP, N_PATHS, seed=2)

fig = ps.make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=['TWAP: Price Impact Decomposition',
                    'Aggressive: Price Impact Decomposition'])

t_ax = np.arange(N + 1)
for paths, perm, temp, col in [(twap_paths, twap_perm, twap_temp, 1),
                                 (aggr_paths, aggr_perm, aggr_temp, 2)]:
    mean_path = paths.mean(axis=0)
    std_path  = paths.std(axis=0)
    fig.add_trace(go.Scatter(
        x=np.concatenate([t_ax, t_ax[::-1]]),
        y=np.concatenate([mean_path + std_path, (mean_path - std_path)[::-1]]),
        fill='toself', fillcolor='rgba(0,212,170,0.12)', line_color='rgba(0,0,0,0)',
        name='±1σ', showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=t_ax, y=mean_path, mode='lines',
        name='Mean price path', line=dict(color='#00d4aa', width=2),
        showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=t_ax, y=-perm, mode='lines',
        name='Permanent impact', line=dict(color='#ff6b6b', dash='dash', width=1.5),
        showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=t_ax, y=-temp, mode='lines',
        name='Temporary impact', line=dict(color='#f0c040', dash='dot', width=1.5),
        showlegend=(col == 1)), row=1, col=col)

fig.update_layout(template='plotly_dark', height=420,
    title='Temporary vs Permanent Market Impact')
fig.update_xaxes(title_text='Period', row=1, col=1)
fig.update_xaxes(title_text='Period', row=1, col=2)
fig.update_yaxes(title_text='Price change ($)', row=1, col=1)
fig.show()

print('Aggressive: high temporary spike early, larger permanent shift')
print('TWAP: smaller temporary impact each period, same permanent impact (same total shares)')

Aggressive: high temporary spike early, larger permanent shift
TWAP: smaller temporary impact each period, same permanent impact (same total shares)


---
## Part 4 | Liquidity Premium

The **liquidity premium** is the excess return investors earn for holding illiquid assets — compensation for:
- Being unable to exit quickly without price impact
- Wider spreads on every trade
- Slower price discovery (inefficiency premium)

We demonstrate using our four-stock universe: sort by spread and compare **gross** vs **net-of-cost** returns, and estimate the premium that must be earned to justify the friction.

In [9]:
# ── Gross return, friction, and liquidity premium per stock ───────────────────
HOLD_DAYS   = 20    # rebalance every 20 days (monthly)
ORDER_SIZE  = 1_000 # shares per rebalance

lp_rows = []
for name, p in UNIVERSE.items():
    px     = p['price']
    trades_per_year = ANNUAL / HOLD_DAYS
    notional = ORDER_SIZE * px

    # Annual gross return (simulated)
    gross_ann = p['mu'] * ANNUAL

    # Annual cost: spread + impact, per round-trip, × turnover
    spread_rt  = (p['spread_bps'] / 2) / 10_000   # one-way
    impact_rt  = SQRT_ETA * p['vol'] * np.sqrt(ORDER_SIZE / p['adv'])  # one-way
    cost_per_rt = (spread_rt + impact_rt) * 2      # round-trip
    annual_cost = cost_per_rt * trades_per_year

    net_ann     = gross_ann - annual_cost
    sr_gross    = gross_ann / (p['vol'] * ANNUAL**0.5)
    sr_net      = net_ann   / (p['vol'] * ANNUAL**0.5)

    lp_rows.append(dict(
        stock=name,
        spread_bps=p['spread_bps'],
        gross_ret=gross_ann * 100,
        annual_cost=annual_cost * 100,
        net_ret=net_ann * 100,
        sr_gross=sr_gross,
        sr_net=sr_net,
        premium_bps=annual_cost * 10_000 / trades_per_year   # required premium per trade in bps
    ))

lp_df = pd.DataFrame(lp_rows).set_index('stock')
print(f'Monthly rebalance ({HOLD_DAYS}-day hold) | {ORDER_SIZE} shares | {int(ANNUAL/HOLD_DAYS)} trades/yr\n')
print(lp_df[['spread_bps','gross_ret','annual_cost','net_ret','sr_gross','sr_net']]
      .rename(columns={'spread_bps':'Spread(bps)','gross_ret':'GrossRet%',
                        'annual_cost':'AnnCost%','net_ret':'NetRet%',
                        'sr_gross':'SR_gross','sr_net':'SR_net'})
      .to_string(float_format='{:.3f}'.format))

Monthly rebalance (20-day hold) | 1000 shares | 12 trades/yr

             Spread(bps)  GrossRet%  AnnCost%  NetRet%  SR_gross  SR_net
stock                                                                   
LC_Liquid              2      5.040     0.308    4.732     0.317   0.298
LC_Illiquid            8      7.560     1.183    6.377     0.397   0.335
SC_Liquid             15     10.080     2.260    7.820     0.353   0.274
SC_Illiquid           35     15.120     5.287    9.833     0.433   0.282


In [10]:
# ── Plot liquidity premium ─────────────────────────────────────────────────────
fig = ps.make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=['Gross vs Net Annual Return by Liquidity Tier',
                    'SR Gross vs Net (Friction Drag on Risk-Adjusted Returns)'])

stocks = lp_df.index.tolist()
fig.add_trace(go.Bar(x=stocks, y=lp_df['gross_ret'], name='Gross Return %',
    marker_color='#00d4aa', opacity=0.8), row=1, col=1)
fig.add_trace(go.Bar(x=stocks, y=lp_df['net_ret'], name='Net Return %',
    marker_color='#ff6b6b', opacity=0.85), row=1, col=1)
fig.add_trace(go.Scatter(x=stocks, y=lp_df['annual_cost'],
    mode='markers+lines', name='Annual Cost %',
    marker=dict(color='#f0c040', size=10, symbol='diamond'),
    line=dict(color='#f0c040', dash='dot')), row=1, col=1)

x_sr = np.arange(len(stocks))
fig.add_trace(go.Bar(x=stocks, y=lp_df['sr_gross'], name='SR Gross',
    marker_color='#00d4aa', opacity=0.8), row=1, col=2)
fig.add_trace(go.Bar(x=stocks, y=lp_df['sr_net'], name='SR Net',
    marker_color='#ff6b6b', opacity=0.85), row=1, col=2)
fig.add_hline(y=0, line_color='grey', line_dash='dot', row=1, col=2)

fig.update_layout(template='plotly_dark', height=440, barmode='group',
    title='Liquidity Premium: Gross vs Net Returns Across Liquidity Tiers')
fig.update_yaxes(title_text='Annual Return (%)', row=1, col=1)
fig.update_yaxes(title_text='Sharpe Ratio', row=1, col=2)
fig.show()

# Required alpha premium for illiquid stocks to break even vs liquid
lc_net = lp_df.loc['LC_Liquid', 'net_ret']
for name in ['LC_Illiquid', 'SC_Liquid', 'SC_Illiquid']:
    excess = lp_df.loc[name, 'net_ret'] - lc_net
    print(f'{name}: net return premium vs LC_Liquid = {excess:+.2f}%/yr')

LC_Illiquid: net return premium vs LC_Liquid = +1.65%/yr
SC_Liquid: net return premium vs LC_Liquid = +3.09%/yr
SC_Illiquid: net return premium vs LC_Liquid = +5.10%/yr


In [11]:
# ── Cumulative wealth: gross vs net equity curves ─────────────────────────────
fig = go.Figure()
line_styles = [dict(width=2), dict(width=2, dash='dot')]
stock_colors = {'LC_Liquid': 'cornflowerblue', 'LC_Illiquid': '#00d4aa',
                'SC_Liquid': '#f0c040',         'SC_Illiquid': '#ff6b6b'}

for name, p in UNIVERSE.items():
    r = ret_df[name]
    # Gross
    eq_gross = (1 + r.fillna(0)).cumprod()
    fig.add_trace(go.Scatter(x=dates, y=eq_gross, mode='lines',
        name=f'{name} gross',
        line=dict(color=stock_colors[name], width=2)))

    # Net: deduct round-trip cost every HOLD_DAYS bars
    spread_rt = (p['spread_bps'] / 2) / 10_000
    impact_rt = SQRT_ETA * p['vol'] * np.sqrt(ORDER_SIZE / p['adv'])
    cost_rt   = (spread_rt + impact_rt) * 2
    cost_series = pd.Series(0.0, index=dates)
    cost_series.iloc[::HOLD_DAYS] = cost_rt
    net_r = r - cost_series
    eq_net = (1 + net_r.fillna(0)).cumprod()
    fig.add_trace(go.Scatter(x=dates, y=eq_net, mode='lines',
        name=f'{name} net',
        line=dict(color=stock_colors[name], width=1.5, dash='dot')))

fig.add_hline(y=1, line_color='grey', line_dash='dot')
fig.update_layout(template='plotly_dark', height=450,
    title=f'Gross vs Net Equity — {HOLD_DAYS}-day Rebalance | {ORDER_SIZE} shares',
    xaxis_title='Date', yaxis_title='Portfolio Value')
fig.show()

---
## Part 5 | Optimal Urgency & Capacity

**Optimal urgency** is the trade-off between:
- **Trading fast** (high POV%) → high market impact, low alpha decay  
- **Trading slow** (low POV%) → low market impact, high alpha decay (signal loses value while waiting)

We model alpha as an exponential decay with half-life $\tau$:

$$\alpha(t) = \alpha_0 \cdot e^{-\ln(2) \cdot t / \tau}$$

The optimal POV minimises **net alpha** = gross alpha – market impact cost.

In [12]:
# ── Alpha decay vs market impact: sweep POV across stocks ─────────────────────
ALPHA0_BPS  = 20.0         # initial alpha edge at signal in bps
TAU_DAYS    = 3.0          # alpha half-life in trading days
ADV_BARS    = 1            # 1 bar = 1 day for simplicity

pov_range = np.linspace(0.005, 0.40, 200)   # 0.5% to 40% of ADV

def net_alpha(pov, adv, vol, alpha0_bps, tau, eta):
    """
    Net alpha = gross alpha captured (accounting for decay) - market impact cost.
    All in bps.
    """
    # Time to fill at given POV: 1 / pov days (fill 100% of position)
    fill_time      = 1.0 / pov                                 # days to complete
    alpha_captured = alpha0_bps * (1 - np.exp(-np.log(2) * fill_time / tau))
    # Average alpha per day (integral / fill_time) × fill_time = integral
    alpha_integral = alpha0_bps * tau / np.log(2) * (1 - np.exp(-np.log(2) * fill_time / tau))

    # Market impact: square-root, one-way, in bps
    impact_bps = eta * vol * np.sqrt(pov) * 10_000

    # Net: alpha captured minus impact
    return alpha0_bps - impact_bps, impact_bps, alpha0_bps * np.exp(-np.log(2) * fill_time / tau)

fig = ps.make_subplots(rows=2, cols=2, vertical_spacing=0.12, horizontal_spacing=0.10,
    subplot_titles=[f'LC_Liquid (spread={UNIVERSE["LC_Liquid"]["spread_bps"]}bps)',
                    f'LC_Illiquid (spread={UNIVERSE["LC_Illiquid"]["spread_bps"]}bps)',
                    f'SC_Liquid (spread={UNIVERSE["SC_Liquid"]["spread_bps"]}bps)',
                    f'SC_Illiquid (spread={UNIVERSE["SC_Illiquid"]["spread_bps"]}bps)'])

positions = [(1,1),(1,2),(2,1),(2,2)]
optimal_povs = {}

for (name, p), (r, c) in zip(UNIVERSE.items(), positions):
    nets, impacts, residuals = [], [], []
    for pov in pov_range:
        n, imp, res = net_alpha(pov, p['adv'], p['vol'], ALPHA0_BPS, TAU_DAYS, SQRT_ETA)
        nets.append(n)
        impacts.append(imp)
        residuals.append(res)

    nets     = np.array(nets)
    impacts  = np.array(impacts)

    best_idx = np.argmax(nets)
    opt_pov  = pov_range[best_idx]
    optimal_povs[name] = opt_pov

    fig.add_trace(go.Scatter(x=pov_range*100, y=np.full_like(pov_range, ALPHA0_BPS),
        mode='lines', name='Gross alpha', line=dict(color='white', dash='dot', width=1),
        showlegend=(r==1 and c==1)), row=r, col=c)
    fig.add_trace(go.Scatter(x=pov_range*100, y=nets,
        mode='lines', name='Net alpha (gross - impact)',
        line=dict(color='#00d4aa', width=2), showlegend=(r==1 and c==1)), row=r, col=c)
    fig.add_trace(go.Scatter(x=pov_range*100, y=impacts,
        mode='lines', name='Market impact',
        line=dict(color='#ff6b6b', width=1.5), showlegend=(r==1 and c==1)), row=r, col=c)
    fig.add_vline(x=opt_pov * 100, line_color='#f0c040', line_dash='dash', row=r, col=c)
    fig.add_hline(y=0, line_color='grey', line_dash='dot', row=r, col=c)

fig.update_layout(template='plotly_dark', height=580,
    title=f'Optimal Urgency: Net Alpha vs Participation Rate  (α₀={ALPHA0_BPS}bps, τ={TAU_DAYS}d)')
for r, c in positions:
    fig.update_xaxes(title_text='POV (%)', row=r, col=c)
    fig.update_yaxes(title_text='bps', row=r, col=c)
fig.show()

print('Optimal POV by stock:')
for name, pov in optimal_povs.items():
    print(f'  {name:<14}: {pov*100:.1f}% of ADV  ({UNIVERSE[name]["adv"] * pov:,.0f} shares/day)')

Optimal POV by stock:
  LC_Liquid     : 0.5% of ADV  (10,000 shares/day)
  LC_Illiquid   : 0.5% of ADV  (1,500 shares/day)
  SC_Liquid     : 0.5% of ADV  (750 shares/day)
  SC_Illiquid   : 0.5% of ADV  (200 shares/day)


In [13]:
# ── Capacity curve: max AUM before alpha is fully consumed ────────────────────
AUM_RANGE   = np.logspace(4, 8, 80)    # $10k to $100M

fig = go.Figure()

for name, p in UNIVERSE.items():
    net_alphas_by_aum = []
    for aum in AUM_RANGE:
        shares  = aum / p['price']
        pov     = shares / p['adv']
        net, _, _ = net_alpha(pov, p['adv'], p['vol'], ALPHA0_BPS, TAU_DAYS, SQRT_ETA)
        net_alphas_by_aum.append(net)

    net_arr = np.array(net_alphas_by_aum)

    # Capacity: where net alpha first crosses zero
    zero_cross = np.where(net_arr <= 0)[0]
    capacity   = AUM_RANGE[zero_cross[0]] if len(zero_cross) else AUM_RANGE[-1]

    fig.add_trace(go.Scatter(
        x=AUM_RANGE / 1e6, y=net_arr,
        mode='lines', name=f'{name} (cap ~${capacity/1e6:.1f}M)',
        line=dict(width=2)))

    if len(zero_cross):
        fig.add_shape(type='line', xref='x', yref='paper',
            x0=capacity/1e6, x1=capacity/1e6, y0=0, y1=1,
            line=dict(dash='dot', width=1, color='grey'))

fig.add_hline(y=0, line_color='#ff6b6b', line_dash='dash',
    annotation_text='Alpha consumed', annotation_position='right')

fig.update_layout(
    template='plotly_dark', height=440,
    title=f'Strategy Capacity — Net Alpha vs AUM  (α₀={ALPHA0_BPS}bps signal)',
    xaxis_title='AUM ($M)',
    yaxis_title='Net Alpha (bps)',
    xaxis_type='log',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

print('Strategy capacity (net alpha → 0):')
for name, p in UNIVERSE.items():
    net_arr = []
    for aum in AUM_RANGE:
        shares = aum / p['price']
        pov    = shares / p['adv']
        n, _, _ = net_alpha(pov, p['adv'], p['vol'], ALPHA0_BPS, TAU_DAYS, SQRT_ETA)
        net_arr.append(n)
    zc = np.where(np.array(net_arr) <= 0)[0]
    cap = AUM_RANGE[zc[0]] if len(zc) else AUM_RANGE[-1]
    print(f'  {name:<14}: ${cap/1e6:.2f}M')

Strategy capacity (net alpha → 0):
  LC_Liquid     : $100.00M
  LC_Illiquid   : $100.00M
  SC_Liquid     : $8.64M
  SC_Illiquid   : $0.66M


In [14]:
# ── Summary dashboard ──────────────────────────────────────────────────────────
fig = ps.make_subplots(
    rows=2, cols=3, vertical_spacing=0.14, horizontal_spacing=0.08,
    subplot_titles=[
        '1. Cost Decomposition (bps)',
        '2. Cost Model Layers — Equity',
        '3. Execution Frontier (AC)',
        '4. Liquidity Premium: SR Gross vs Net',
        '5. Optimal POV by Stock',
        '5. Capacity (Net Alpha vs AUM)'
    ]
)

# 1 — cost decomposition stacked bar
for col_name, label, color in zip(components, labels, colors):
    fig.add_trace(go.Bar(x=cost_df.index, y=cost_df[col_name],
        name=label, marker_color=color, showlegend=False), row=1, col=1)

# 2 — equity curves by cost layer
for pnl, color, lab in zip([raw_pnl, pnl_l4], ['white', '#ff6b6b'], ['Raw', 'L4 +Impact']):
    fig.add_trace(go.Scatter(x=dates, y=equity(pnl), mode='lines',
        name=lab, line=dict(color=color, width=1.5), showlegend=False), row=1, col=2)

# 3 — AC frontier
fig.add_trace(go.Scatter(x=risks_arr / X0 * 100, y=costs_arr / X0 * 10_000,
    mode='lines', line=dict(color='#00d4aa', width=2), showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=[twap_risk/X0*100], y=[twap_cost/X0*10_000],
    mode='markers', marker=dict(color='#f0c040', size=10, symbol='star'),
    showlegend=False), row=1, col=3)

# 4 — SR gross vs net
fig.add_trace(go.Bar(x=lp_df.index, y=lp_df['sr_gross'], name='SR Gross',
    marker_color='#00d4aa', opacity=0.8, showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=lp_df.index, y=lp_df['sr_net'], name='SR Net',
    marker_color='#ff6b6b', opacity=0.85, showlegend=False), row=2, col=1)

# 5 — optimal POV
fig.add_trace(go.Bar(
    x=list(optimal_povs.keys()),
    y=[v * 100 for v in optimal_povs.values()],
    marker_color='cornflowerblue', showlegend=False), row=2, col=2)

# 6 — capacity
for name, p in UNIVERSE.items():
    net_arr_c = []
    for aum in AUM_RANGE:
        pov = (aum / p['price']) / p['adv']
        n, _, _ = net_alpha(pov, p['adv'], p['vol'], ALPHA0_BPS, TAU_DAYS, SQRT_ETA)
        net_arr_c.append(n)
    fig.add_trace(go.Scatter(x=AUM_RANGE/1e6, y=net_arr_c, mode='lines',
        line=dict(width=1.5), showlegend=False), row=2, col=3)
fig.add_hline(y=0, line_color='#ff6b6b', line_dash='dot', row=2, col=3)

fig.update_layout(template='plotly_dark', height=660, barmode='group',
    title='Day 17 Summary: Transaction Costs & Liquidity Premium')
fig.update_xaxes(title_text='AUM ($M)', type='log', row=2, col=3)
fig.update_xaxes(title_text='Timing Risk (%)', row=1, col=3)
fig.update_xaxes(title_text='POV (%)', row=2, col=2)
fig.show()

---
## Summary — The Execution Realism Checklist

| # | Concept | Implemented in this notebook |
|---|---------|------------------------------|
| 1 | Cost taxonomy | Explicit (commission, fees) vs implicit (spread, impact) decomposition |
| 2 | Dynamic spreads | Spread widens with realised vol; different tiers per stock |
| 3 | Flat fee & % model | Baseline Level 1 & 2 cost layers |
| 4 | Bid-ask spread model | Dynamic half-spread applied on every trade |
| 5 | Square-root impact | Kyle/Barra market impact scaled to ADV & vol |
| 6 | Almgren-Chriss framework | Temporary vs permanent impact; execution frontier |
| 7 | Liquidity premium | Gross vs net returns across 4 liquidity tiers |
| 8 | Optimal urgency | POV sweep to maximise net alpha (alpha decay vs impact) |
| 9 | Capacity analysis | AUM ceiling where net alpha crosses zero per stock |
| 10 | Model selection guide | LC liquid → flat fee; institutional → square-root; HFT → microstructure |

> *"Liquidity is the lifeblood of markets. Understanding its costs and premiums is the key to sustainable quantitative trading."*